# Advanced ML Model (Random Forest) 
Run Setup to generate files:


In [ ]:
!pip install librosa numpy pandas matplotlib scikit-learn tensorflow seaborn

import os
import librosa
import librosa.display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.io.wavfile as wavfile
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')

# 1. GENERATE DATASET WITH CSV METADATA
def create_dataset():
    classes = {'crying': 1.5, 'hungry': 1.0, 'discomfort': 0.8, 'sleeping': 0.2}
    base_dir = "data/raw"
    os.makedirs('data', exist_ok=True)
    os.makedirs(base_dir, exist_ok=True)
    metadata = []
    
    for cls, freq_mod in classes.items():
        cls_dir = os.path.join(base_dir, cls)
        os.makedirs(cls_dir, exist_ok=True)
        for i in range(25):
            t = np.linspace(0, 2.0, int(22050 * 2.0))
            signal = np.sin(2 * np.pi * 440 * freq_mod * t) + np.random.normal(0, 0.5, len(t))
            signal = signal / np.max(np.abs(signal))
            signal += np.random.normal(0, 0.1, len(signal))
            signal = np.int16(signal * 32767)
            
            filepath = os.path.join(cls_dir, f"sample_{i:03d}.wav")
            wavfile.write(filepath, 22050, signal)
            metadata.append({'filename': filepath, 'class': cls})
            
    pd.DataFrame(metadata).to_csv('data/metadata.csv', index=False)
    print("Dataset generation complete. Mappings successfully saved to 'data/metadata.csv'.")

if not os.path.exists('data/metadata.csv'):
    create_dataset()

# 2. FEATURE EXTRACTION FUNCTIONS
def extract_ml_features(audio, sr=22050):
    mfcc = np.mean(librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40).T, axis=0)
    zcr = np.mean(librosa.feature.zero_crossing_rate(y=audio).T, axis=0)
    chroma = np.mean(librosa.feature.chroma_stft(S=np.abs(librosa.stft(audio)), sr=sr).T, axis=0)
    return np.hstack([mfcc, zcr, chroma])

def extract_dl_features(audio, sr=22050, max_pad_len=100):
    mel_spec_db = librosa.power_to_db(librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128), ref=np.max)
    if mel_spec_db.shape[1] > max_pad_len:
        return mel_spec_db[:, :max_pad_len]
    else:
        pad_width = max_pad_len - mel_spec_db.shape[1]
        return np.pad(mel_spec_db, pad_width=((0,0), (0, pad_width)), mode='constant')

def load_dataset(feature_type='ml'):
    df = pd.read_csv('data/metadata.csv')
    classes = sorted(df['class'].unique())
    class_map = {label: idx for idx, label in enumerate(classes)}
    data, labels = [], []
    
    for _, row in df.iterrows():
        audio, sr = librosa.load(row['filename'], sr=22050)
        # Add original and augmented
        for a in [audio, audio + 0.005 * np.random.randn(len(audio))]:
            feats = extract_ml_features(a, sr) if feature_type == 'ml' else extract_dl_features(a, sr)
            data.append(feats)
            labels.append(class_map[row['class']])
            
    return np.array(data), np.array(labels), class_map


In [ ]:
from sklearn.ensemble import RandomForestClassifier

print('Loading ML Features...')
X, y, class_mapping = load_dataset('ml')
class_names = [k for k, v in sorted(class_mapping.items(), key=lambda item: item[1])]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print('Training Random Forest...')
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(f'Advanced RF Accuracy: {accuracy_score(y_test, y_pred):.4f}')

sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.title('Advanced ML Confusion Matrix')
plt.show()


In [ ]:
from sklearn.svm import SVC

print('Training SVM (RBF)...')
svm = SVC(kernel='rbf', probability=True, random_state=42)
svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)
print(f'SVM Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}')

plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_svm), annot=True, fmt='d', cmap='Oranges',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Advanced SVM Confusion Matrix')
plt.tight_layout()
plt.show()


In [ ]:
!pip install xgboost -q
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

print('Training XGBoost...')
xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42, n_jobs=-1)

param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6],
    'learning_rate': [0.1, 0.05],
    'subsample': [0.8, 1.0]
}
grid_xgb = GridSearchCV(xgb, param_grid_xgb, cv=3, scoring='f1_weighted', n_jobs=-1, verbose=1)
grid_xgb.fit(X_train, y_train)
best_xgb = grid_xgb.best_estimator_
print(f'Best XGBoost Params: {grid_xgb.best_params_}')

y_pred_xgb = best_xgb.predict(X_test)
print(f'XGBoost Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}')
print(classification_report(y_test, y_pred_xgb, target_names=class_names))

plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_xgb), annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('XGBoost Confusion Matrix')
plt.tight_layout()
plt.show()

# Feature importance plot
feat_imp = best_xgb.feature_importances_
top_n = 20
top_idx = feat_imp.argsort()[-top_n:][::-1]
plt.figure(figsize=(10, 4))
plt.bar(range(top_n), feat_imp[top_idx])
plt.xticks(range(top_n), [f'f{i}' for i in top_idx], rotation=45)
plt.title('XGBoost — Top 20 Feature Importances')
plt.tight_layout()
plt.show()


In [ ]:
# ---- ML Models Comparison Summary ----
from sklearn.metrics import f1_score, precision_score, recall_score

model_results = {
    'Random Forest': (y_pred, y_test),
    'SVM (RBF)':     (y_pred_svm, y_test),
    'XGBoost':       (y_pred_xgb, y_test),
}

summary = []
for name, (pred, true) in model_results.items():
    summary.append({
        'Model': name,
        'Accuracy': accuracy_score(true, pred),
        'Precision': precision_score(true, pred, average='weighted', zero_division=0),
        'Recall': recall_score(true, pred, average='weighted', zero_division=0),
        'F1-Score': f1_score(true, pred, average='weighted', zero_division=0),
    })

df_summary = pd.DataFrame(summary).set_index('Model')
print(df_summary.to_string(float_format='{:.4f}'.format))

# Bar chart
ax = df_summary[['Accuracy', 'F1-Score']].plot(kind='bar', figsize=(8, 4), colormap='tab10', rot=15)
ax.set_ylim(0, 1.1)
ax.set_title('Infant Cry Recognition — ML Models Comparison (Kaggle Dataset)')
ax.set_ylabel('Score')
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=2, fontsize=8)
plt.tight_layout()
plt.show()
